# 第十章：PCA 与 DDcGAN 重点专题

## 降维的数学之美与对抗生成的巅峰设计

本章是两个重点专题的深度讲解。**第一部分**从数学本质推导 PCA，用 NumPy 手写实现，
结合可视化理解降维；**第二部分**深入剖析 DDcGAN——双判别器条件对抗网络的原理、
架构和完整实现。每一个公式都有推导，每一行代码都有注释。


---

# 第一部分：主成分分析 (PCA)

## 7.1 什么是 PCA？

**主成分分析 (Principal Component Analysis)** 是最经典的线性降维方法，由 Karl Pearson 于 1901 年提出。
它的核心思想是：**找到数据方差最大的方向（主成分），将数据投影到这些方向上，在保留最多信息的前提下降低维度。**

### 直观理解

想象你在 3D 空间中看一团点云。如果这些点几乎分布在一个平面上，你可以把这个平面旋转到"正面",
然后用 2D 坐标描述所有点——这就是 PCA 在做的事。

### 数学目标

给定数据中心化后的数据矩阵 $\mathbf{X} \in \mathbb{R}^{n \times d}$（$n$ 个样本，$d$ 个特征），
找到投影矩阵 $\mathbf{W} \in \mathbb{R}^{d \times k}$（$k < d$），使得投影后的数据方差最大：

$$
\mathbf{W}^* = \arg\max_{\mathbf{W}} \text{Tr}(\mathbf{W}^T \mathbf{X}^T \mathbf{X} \mathbf{W}), \quad \text{s.t. } \mathbf{W}^T \mathbf{W} = \mathbf{I}
$$

此优化问题的解是：$\mathbf{W}$ 的列是 $\mathbf{X}^T \mathbf{X}$（即协方差矩阵的 $n$ 倍）的**前 $k$ 个最大特征值对应的特征向量**。


## 7.2 PCA 的数学推导

### Step 1: 数据中心化

$$
\bar{\mathbf{x}} = \frac{1}{n} \sum_{i=1}^{n} \mathbf{x}_i, \quad \tilde{\mathbf{x}}_i = \mathbf{x}_i - \bar{\mathbf{x}}
$$

### Step 2: 计算协方差矩阵

$$
\mathbf{C} = \frac{1}{n-1} \tilde{\mathbf{X}}^T \tilde{\mathbf{X}}
$$

$\mathbf{C}_{ij}$ 表示第 $i$ 个特征和第 $j$ 个特征之间的协方差：
- $\mathbf{C}_{ii} = \text{Var}(X_i)$：特征 $i$ 的方差
- $\mathbf{C}_{ij} = \text{Cov}(X_i, X_j)$：特征 $i$ 和 $j$ 的协方差

### Step 3: 特征值分解

$$
\mathbf{C} = \mathbf{V} \mathbf{\Lambda} \mathbf{V}^T
$$

- $\mathbf{\Lambda} = \text{diag}(\lambda_1, \ldots, \lambda_d)$：特征值（按降序排列，$\lambda_1 \ge \lambda_2 \ge \ldots \ge \lambda_d$）
- $\mathbf{V} = [\mathbf{v}_1, \ldots, \mathbf{v}_d]$：对应的特征向量

**特征值 $\lambda_i$ = 数据在方向 $\mathbf{v}_i$ 上的方差**。

### Step 4: 选择前 $k$ 个主成分

保留前 $k$ 个最大特征值对应的特征向量，构成投影矩阵 $\mathbf{W} = [\mathbf{v}_1, \ldots, \mathbf{v}_k]$。

降维后的数据：
$$
\mathbf{Z} = \tilde{\mathbf{X}} \mathbf{W}
$$

### 方差解释率

第 $i$ 个主成分解释的方差比例：
$$
\text{Explained Variance Ratio}_i = \frac{\lambda_i}{\sum_{j=1}^{d} \lambda_j}
$$

前 $k$ 个主成分的累积解释方差：
$$
\text{Cumulative} = \frac{\sum_{i=1}^{k} \lambda_i}{\sum_{j=1}^{d} \lambda_j}
$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits, load_iris, make_blobs
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# === PCA 从零实现 ===
class PCA:
    """主成分分析 — 基于特征值分解的手工实现"""
    
    def __init__(self, n_components=None):
        self.n_components = n_components
        self.components_ = None      # 主成分方向 (k × d)
        self.explained_variance_ = None
        self.explained_variance_ratio_ = None
        self.mean_ = None
    
    def fit(self, X):
        """
        X: (n_samples, n_features)
        
        步骤:
        1. 中心化
        2. 计算协方差矩阵
        3. 特征值分解
        4. 选择前 k 个主成分
        """
        n, d = X.shape
        
        # Step 1: 中心化
        self.mean_ = X.mean(axis=0)
        X_centered = X - self.mean_
        
        # Step 2: 协方差矩阵 C = (1/(n-1)) X^T X
        C = (X_centered.T @ X_centered) / (n - 1)
        
        # Step 3: 特征值分解
        eigenvalues, eigenvectors = np.linalg.eigh(C)
        
        # eigh 返回升序，反转为降序
        idx = np.argsort(eigenvalues)[::-1]
        eigenvalues = eigenvalues[idx]
        eigenvectors = eigenvectors[:, idx]
        
        # Step 4: 选择前 k 个主成分
        k = self.n_components if self.n_components else d
        self.components_ = eigenvectors[:, :k].T  # (k, d)
        self.explained_variance_ = eigenvalues[:k]
        self.explained_variance_ratio_ = eigenvalues[:k] / eigenvalues.sum()
        
        return self
    
    def transform(self, X):
        """降维: Z = (X - mean) @ W^T"""
        X_centered = X - self.mean_
        return X_centered @ self.components_.T
    
    def inverse_transform(self, Z):
        """重建: X_recon = Z @ W + mean"""
        return Z @ self.components_ + self.mean_

# === 测试：二维数据降维到一维 ===
np.random.seed(42)
# 生成有相关性的二维数据
mean = [3, 7]
cov = [[2.0, 1.5], [1.5, 1.0]]  # 正相关
X_test = np.random.multivariate_normal(mean, cov, 200)

pca = PCA(n_components=2)
pca.fit(X_test)

print("=" * 50)
print("PCA 结果分析")
print("=" * 50)
print(f"特征值: {pca.explained_variance_}")
print(f"方差解释率: {pca.explained_variance_ratio_}")
print(f"累积解释率: {pca.explained_variance_ratio_.cumsum()}")
print(f"\n第一主成分方向: {pca.components_[0]}")
print(f"第二主成分方向: {pca.components_[1]}")
print(f"两主成分正交: {np.dot(pca.components_[0], pca.components_[1]):.10f}")


In [ ]:
# === PCA 降维可视化 ===
X_proj = pca.transform(X_test)
X_recon = pca.inverse_transform(X_proj)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 原始数据 + 主成分方向
ax = axes[0]
ax.scatter(X_test[:, 0], X_test[:, 1], alpha=0.5, s=10)
origin = pca.mean_
for i, (comp, var) in enumerate(zip(pca.components_, pca.explained_variance_)):
    # 箭头长度 = 2 * sqrt(特征值)（2 个标准差）
    length = 2 * np.sqrt(var)
    ax.arrow(origin[0], origin[1],
             comp[0] * length, comp[1] * length,
             head_width=0.3, head_length=0.3,
             fc=['red', 'orange'][i], ec=['red', 'orange'][i],
             alpha=0.8, linewidth=2,
             label=f'PC{i+1} ({pca.explained_variance_ratio_[i]:.1%})')
ax.set_title('Original Data + Principal Components')
ax.set_xlabel('x1'); ax.set_ylabel('x2')
ax.legend(); ax.grid(True, alpha=0.3); ax.axis('equal')

# 降维后的一维分布
ax = axes[1]
ax.scatter(X_proj[:, 0], np.zeros_like(X_proj[:, 0]), alpha=0.5, s=10)
ax.hist(X_proj[:, 0], bins=30, alpha=0.5, density=True, color='green')
ax.set_title('Projected to PC1 (1D)')
ax.set_xlabel('PC1'); ax.set_yticks([])

# 重建 vs 原始
ax = axes[2]
ax.scatter(X_test[:, 0], X_test[:, 1], alpha=0.3, s=10, label='Original')
ax.scatter(X_recon[:, 0], X_recon[:, 1], alpha=0.7, s=10, label='Reconstructed')
# 连几条线展示投影方向
for i in range(20):
    ax.plot([X_test[i, 0], X_recon[i, 0]],
            [X_test[i, 1], X_recon[i, 1]], 'gray', alpha=0.3, lw=0.5)
ax.set_title('Original vs Reconstructed (with PC1 only)')
ax.legend(); ax.grid(True, alpha=0.3); ax.axis('equal')

plt.tight_layout()
plt.savefig('/home/yisan/ai-learning/notebooks/pca_2d_demo.png', dpi=100, bbox_inches='tight')
plt.show()

# 重建误差
recon_error = np.mean((X_test - X_recon) ** 2)
print(f"\n仅用 PC1 重建的均方误差: {recon_error:.4f}")
print(f"PC1 损失的信息比例: {1 - pca.explained_variance_ratio_[0]:.2%}")


## 7.3 PCA 在高维数据上的应用

### 手写数字 (Digits) 降维

Digits 数据集：8×8 灰度手写数字图片（64 维）。我们用它演示 PCA 降维和重建。


In [ ]:
# === Digits 数据集 PCA ===
digits = load_digits()
X_digits = digits.data      # (1797, 64)
y_digits = digits.target

print(f"数据形状: {X_digits.shape}")
print(f"值范围: [{X_digits.min():.1f}, {X_digits.max():.1f}]")

# PCA 降维
pca_digits = PCA()
pca_digits.fit(X_digits)

# 方差解释曲线
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.bar(range(1, 21), pca_digits.explained_variance_ratio_[:20], alpha=0.7)
cumsum = pca_digits.explained_variance_ratio_.cumsum()
ax2 = ax.twinx()
ax2.plot(range(1, 21), cumsum[:20], 'ro-', markersize=4, linewidth=2)
ax2.set_ylabel('Cumulative Explained Variance', color='red')
ax2.set_ylim(0, 1.05)

# 标注关键累积值
for k in [2, 5, 10, 16, 20]:
    ax2.annotate(f'{cumsum[k-1]:.1%}',
                xy=(k, cumsum[k-1]),
                xytext=(k+0.5, cumsum[k-1]-0.05),
                fontsize=8, color='red')

ax.set_xlabel('Principal Component'); ax.set_ylabel('Explained Variance Ratio')
ax.set_title('PCA on Digits Dataset (64 dims)')
ax.grid(True, alpha=0.3)

# 2D 投影可视化
ax = axes[1]
Z = pca_digits.transform(X_digits)
scatter = ax.scatter(Z[:, 0], Z[:, 1], c=y_digits, cmap='tab10',
                     alpha=0.6, s=15)
ax.set_xlabel(f'PC1 ({pca_digits.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca_digits.explained_variance_ratio_[1]:.1%})')
ax.set_title('Digits Projected to First 2 PCs')
plt.colorbar(scatter, ax=ax, label='Digit')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/home/yisan/ai-learning/notebooks/pca_digits.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"前 2 个 PC 解释的方差: {cumsum[1]:.2%}")
print(f"前 10 个 PC 解释的方差: {cumsum[9]:.2%}")
print(f"前 16 个 PC 解释的方差: {cumsum[15]:.2%}")


In [ ]:
# === 用不同数量主成分重建图像 ===
fig, axes = plt.subplots(4, 8, figsize=(16, 8))
n_components_list = [1, 2, 4, 8, 16, 32, 48, 64]
sample_idx = 0  # 第一个数字

for i, n_comp in enumerate(n_components_list):
    # 用 n_comp 个 PC 重建
    pca_temp = PCA(n_components=n_comp)
    pca_temp.fit(X_digits)
    Z = pca_temp.transform(X_digits)
    X_recon = pca_temp.inverse_transform(Z)
    
    # 原始图像
    axes[0, i].imshow(X_digits[sample_idx].reshape(8, 8), cmap='gray')
    axes[0, i].set_title(f'Original', fontsize=9)
    axes[0, i].axis('off')
    
    # 重建图像
    axes[1, i].imshow(X_recon[sample_idx].reshape(8, 8), cmap='gray')
    axes[1, i].set_title(f'{n_comp} PCs', fontsize=9)
    axes[1, i].axis('off')
    
    # 重建误差分布
    error = ((X_digits - X_recon) ** 2).mean(axis=1)
    axes[2, i].hist(error, bins=30, alpha=0.7, color='steelblue')
    axes[2, i].axvline(error[sample_idx], color='red', lw=1.5, linestyle='--')
    axes[2, i].set_title(f'MSE={error.mean():.1f}', fontsize=8)
    axes[2, i].set_xlim(0, 30)
    
    # 累积方差解释
    axes[3, i].bar(range(1, n_comp+1), pca_temp.explained_variance_ratio_, alpha=0.7)
    axes[3, i].set_title(f'Total: {pca_temp.explained_variance_ratio_.sum():.1%}', fontsize=8)
    axes[3, i].set_ylim(0, 0.3)

plt.suptitle('PCA Reconstruction with Different Number of Components', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('/home/yisan/ai-learning/notebooks/pca_reconstruction.png', dpi=100, bbox_inches='tight')
plt.show()


### PCA 的关键洞察

1. **特征值 = 方差**：$\lambda_i$ 越大，该方向包含的信息越多
2. **正交性**：主成分之间线性无关（$\mathbf{v}_i^T \mathbf{v}_j = 0$），去除了特征间的冗余
3. **线性方法**：PCA 只能捕捉线性关系——如果数据分布在曲面上，PCA 效果有限（此时应考虑 t-SNE、UMAP 等非线性方法）
4. **可解释的降维**：每个主成分是原始特征的线性组合，可以分析哪些特征对该成分贡献最大
5. **数据预处理**：PCA 对特征尺度敏感——不同量纲的特征应先标准化（$z = (x-\mu)/\sigma$）


---

# 第二部分：双判别器条件对抗网络 (DDcGAN)

## 7.6 GAN 基础回顾

**生成对抗网络 (Generative Adversarial Network, GAN)** 由 Ian Goodfellow 于 2014 年提出，
开创了生成式建模的对抗训练范式。

### 核心思想：博弈论

GAN 由两个网络组成，进行**极小极大博弈 (Minimax Game)**：

- **生成器 (Generator, $G$)**：从随机噪声 $\mathbf{z} \sim p_z$ 生成假样本 $G(\mathbf{z})$，目标是"骗过"判别器
- **判别器 (Discriminator, $D$)**：判断输入是真实数据还是生成器伪造的，目标是"揪出"假样本

### GAN 的优化目标

$$
\min_G \max_D V(D, G) = \mathbb{E}_{\mathbf{x} \sim p_{\text{data}}} [\log D(\mathbf{x})] + \mathbb{E}_{\mathbf{z} \sim p_z} [\log(1 - D(G(\mathbf{z})))]
$$

- 判别器 $D$ 希望最大化 $V$：对真实数据输出 1，对生成数据输出 0
- 生成器 $G$ 希望最小化 $V$：让 $D(G(z))$ 尽可能接近 1

### 训练交替进行

```
for each iteration:
    1. 训练判别器（固定生成器）：
        - 真实样本 x → D(x) 尽量接近 1
        - 生成样本 G(z) → D(G(z)) 尽量接近 0
    2. 训练生成器（固定判别器）：
        - 生成样本 G(z) → D(G(z)) 尽量接近 1
```

### GAN 的常见问题

- **模式坍塌 (Mode Collapse)**：生成器只产出少数几种样本，缺乏多样性
- **训练不稳定**：判别器和生成器的平衡很难维持
- **梯度消失**：当判别器太强时，生成器收不到有效梯度信号


## 7.7 DDcGAN：双判别器条件对抗网络

**DDcGAN (Dual-Discriminator Conditional GAN)** 是一种改进的条件 GAN 架构，
通过引入**两个结构不同的判别器**来解决 GAN 训练中的核心问题。

### 论文核心思想

传统 GAN 只有一个判别器，容易出现以下问题：
1. 判别器过于强大 → 生成器梯度消失
2. 判别器过于弱小 → 生成器得不到有效反馈
3. 单一判别器可能只关注某些特征（如纹理），忽略其他（如结构）

**DDcGAN 的解决方案：**

> 使用两个判别器——一个关注**全局结构**，一个关注**局部细节**。
> 两个判别器相互补充，迫使生成器同时优化全局和局部质量。

### 架构设计

```
           ┌──────────┐
           │  Noise z │ + [Condition c]
           └────┬─────┘
                ↓
        ┌───────────────┐
        │ Generator (G) │
        └───────┬───────┘
                ↓
          Generated x_fake
                │
        ┌───────┴───────┐
        ↓               ↓
   ┌─────────┐    ┌──────────┐
   │ D_global│    │ D_local  │
   │ (全局)  │    │ (局部)   │
   └────┬────┘    └─────┬────┘
        ↓               ↓
   "真实/假"      "真实/假"
        │               │
        └───────┬───────┘
        Combined Loss → Update G
```

### 两个判别器的角色

| 判别器 | 关注点 | 输入 | 架构特点 |
|--------|--------|------|---------|
| **$D_{\text{global}}$** | 整体结构、全局一致性 | 完整图像 | 较深网络，大感受野 |
| **$D_{\text{local}}$** | 局部细节、纹理质量 | 随机裁剪的图像块 | 较浅网络，局部特征 |

### 条件注入

DDcGAN 是**条件 GAN (cGAN)**——生成器和判别器都以条件 $c$（如类别标签）为额外输入：

$$
\text{生成器：} \quad G(\mathbf{z}, c) \rightarrow \hat{\mathbf{x}}
$$
$$
\text{判别器：} \quad D(\mathbf{x}, c) \rightarrow [0, 1]
$$


## 7.8 DDcGAN 损失函数详解

### 对抗损失 (Adversarial Loss)

DDcGAN 使用 **Hinge Loss** 替代原始 GAN 的交叉熵，训练更稳定：

**判别器损失：**
$$
\mathcal{L}_{D} = \mathbb{E}_{\mathbf{x}\sim p_{\text{real}}} [\max(0, 1 - D(\mathbf{x}))] + \mathbb{E}_{\mathbf{z}\sim p_z} [\max(0, 1 + D(G(\mathbf{z})))]
$$

直觉：
- $D(\mathbf{x}_{\text{real}})$ 应 $\ge 1$（大于 1 不惩罚）
- $D(\mathbf{x}_{\text{fake}})$ 应 $\le -1$（小于 -1 不惩罚）

**生成器损失：**
$$
\mathcal{L}_{G}^{\text{adv}} = -\mathbb{E}_{\mathbf{z}\sim p_z} [D(G(\mathbf{z}))]
$$

生成器希望 $D(G(\mathbf{z}))$ 尽可能大（被判别为真）。

### 双判别器联合损失

DDcGAN 将两个判别器的损失加权组合：

$$
\mathcal{L}_{D}^{\text{total}} = \mathcal{L}_{D_{\text{global}}} + \lambda \cdot \mathcal{L}_{D_{\text{local}}}
$$

$$
\mathcal{L}_{G}^{\text{total}} = \mathcal{L}_{G}^{\text{adv}} + \alpha \cdot \mathcal{L}_{\text{recon}}
$$

其中 $\mathcal{L}_{\text{recon}}$ 是可选的**重建损失 (Reconstruction Loss)**——在图像翻译等任务中使用 $L_1$ 距离。
系数 $\lambda$ 和 $\alpha$ 平衡不同损失项。典型设置：$\lambda = 0.5$，$\alpha = 10$（重建损失权重更大）。

### 为什么双判别器有效？

1. **多角度反馈**：全局判别器提供"形状对不对"的信号；局部判别器提供"纹理真不真"的信号
2. **难度自适应**：当全局判别器变强时，局部判别器可能还弱，生成器仍有梯度来源（反之亦然）
3. **防止模式坍塌**：两个判别器关注不同特征，生成器必须同时满足两者的要求 → 更多样的输出


In [ ]:
# === DDcGAN 核心组件实现（PyTorch）===
import torch
import torch.nn as nn
import torch.nn.functional as F

class Generator(nn.Module):
    """DDcGAN 生成器：噪声 + 条件 → 图像
    使用转置卷积 (ConvTranspose2d) 从低分辨率逐步上采样
    """
    def __init__(self, latent_dim=100, num_classes=10, img_channels=3, feature_dim=64):
        super().__init__()
        self.latent_dim = latent_dim
        self.num_classes = num_classes
        
        # 条件嵌入：将类别标签映射到 latent 空间
        self.label_embedding = nn.Embedding(num_classes, latent_dim)
        
        # 生成器主体：latent_dim*2 (noise + embed) → 4×4×512 → ... → 32×32×3
        self.main = nn.Sequential(
            # 输入: (latent_dim*2) × 1 × 1
            nn.ConvTranspose2d(latent_dim * 2, feature_dim * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(feature_dim * 8),
            nn.ReLU(True),
            # 状态: (512) × 4 × 4
            
            nn.ConvTranspose2d(feature_dim * 8, feature_dim * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_dim * 4),
            nn.ReLU(True),
            # 状态: (256) × 8 × 8
            
            nn.ConvTranspose2d(feature_dim * 4, feature_dim * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_dim * 2),
            nn.ReLU(True),
            # 状态: (128) × 16 × 16
            
            nn.ConvTranspose2d(feature_dim * 2, img_channels, 4, 2, 1, bias=False),
            nn.Tanh()  # 输出范围 [-1, 1]
            # 状态: (3) × 32 × 32
        )
    
    def forward(self, z, labels):
        # 条件嵌入
        c = self.label_embedding(labels).unsqueeze(-1).unsqueeze(-1)
        # 拼接噪声和条件
        z = z.unsqueeze(-1).unsqueeze(-1)
        z_c = torch.cat([z, c], dim=1)  # (batch, latent_dim*2, 1, 1)
        return self.main(z_c)

# ========================================
class GlobalDiscriminator(nn.Module):
    """全局判别器：关注整体图像结构
    使用较深的网络 + 大感受野
    """
    def __init__(self, num_classes=10, img_channels=3, feature_dim=64):
        super().__init__()
        
        # 条件嵌入（投影判别器风格）
        self.label_embedding = nn.Embedding(num_classes, feature_dim * 8)
        
        self.main = nn.Sequential(
            # 输入: 3 × 32 × 32
            nn.Conv2d(img_channels, feature_dim, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # (64) × 16 × 16
            
            nn.Conv2d(feature_dim, feature_dim * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_dim * 2),
            nn.LeakyReLU(0.2, inplace=True),
            # (128) × 8 × 8
            
            nn.Conv2d(feature_dim * 2, feature_dim * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_dim * 4),
            nn.LeakyReLU(0.2, inplace=True),
            # (256) × 4 × 4
            
            nn.Conv2d(feature_dim * 4, feature_dim * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(feature_dim * 8),
            nn.LeakyReLU(0.2, inplace=True),
            # (512) × 1 × 1
        )
    
    def forward(self, img, labels):
        features = self.main(img)  # (batch, 512, 1, 1)
        features = features.view(features.size(0), -1)  # (batch, 512)
        
        # 投影判别器：内积形式注入条件
        label_embed = self.label_embedding(labels)  # (batch, 512)
        output = (features * label_embed).sum(dim=1, keepdim=True)  # (batch, 1)
        return output

# ========================================
class LocalDiscriminator(nn.Module):
    """局部判别器：关注随机图像块的纹理和细节
    使用较浅的网络 + 局部感受野
    """
    def __init__(self, num_classes=10, img_channels=3, feature_dim=32):
        super().__init__()
        
        self.label_embedding = nn.Embedding(num_classes, feature_dim * 4)
        
        self.main = nn.Sequential(
            # 输入: 3 × 16 × 16 (从原图随机裁剪)
            nn.Conv2d(img_channels, feature_dim, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # (32) × 8 × 8
            
            nn.Conv2d(feature_dim, feature_dim * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_dim * 2),
            nn.LeakyReLU(0.2, inplace=True),
            # (64) × 4 × 4
            
            nn.Conv2d(feature_dim * 2, feature_dim * 4, 4, 1, 0, bias=False),
            nn.BatchNorm2d(feature_dim * 4),
            nn.LeakyReLU(0.2, inplace=True),
            # (128) × 1 × 1
        )
    
    def forward(self, img_patch, labels):
        features = self.main(img_patch)
        features = features.view(features.size(0), -1)
        label_embed = self.label_embedding(labels)
        output = (features * label_embed).sum(dim=1, keepdim=True)
        return output


In [ ]:
# === DDcGAN 损失函数 ===
class DDcGANLoss:
    """DDcGAN Hinge Loss + 双判别器加权"""
    
    def __init__(self, lambda_local=0.5):
        self.lambda_local = lambda_local
    
    def discriminator_loss(self, D_global_out_real, D_global_out_fake,
                            D_local_out_real, D_local_out_fake):
        """
        Hinge Loss for both discriminators
        D_real 应 >= 1, D_fake 应 <= -1
        """
        # 全局判别器
        loss_D_global_real = F.relu(1.0 - D_global_out_real).mean()
        loss_D_global_fake = F.relu(1.0 + D_global_out_fake).mean()
        loss_D_global = loss_D_global_real + loss_D_global_fake
        
        # 局部判别器
        loss_D_local_real = F.relu(1.0 - D_local_out_real).mean()
        loss_D_local_fake = F.relu(1.0 + D_local_out_fake).mean()
        loss_D_local = loss_D_local_real + loss_D_local_fake
        
        return loss_D_global + self.lambda_local * loss_D_local
    
    def generator_loss(self, D_global_out_fake, D_local_out_fake):
        """生成器希望 D(G(z)) 尽可能大"""
        loss_G_global = -D_global_out_fake.mean()
        loss_G_local = -D_local_out_fake.mean()
        return loss_G_global + self.lambda_local * loss_G_local


In [ ]:
# === DDcGAN 训练 Demo：在 MNIST 上生成手写数字 ===
# 使用简化版（灰度图 1 通道）训练少量 epoch 演示

def train_ddcgan_demo(epochs=10, batch_size=64, latent_dim=100, device='cpu'):
    """DDcGAN 在 MNIST 上的训练演示"""
    from torchvision import datasets, transforms
    
    # 数据加载
    transform = transforms.Compose([
        transforms.Resize(32),
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5])  # → [-1, 1]
    ])
    
    dataset = datasets.MNIST('/home/yisan/ai-learning/data', train=True,
                              download=True, transform=transform)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    # 模型（灰度图 1 channel）
    G = Generator(latent_dim=latent_dim, num_classes=10, img_channels=1).to(device)
    D_global = GlobalDiscriminator(num_classes=10, img_channels=1).to(device)
    D_local = LocalDiscriminator(num_classes=10, img_channels=1).to(device)
    
    criterion = DDcGANLoss(lambda_local=0.5)
    
    # 优化器
    opt_G = optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
    opt_D = optim.Adam(
        list(D_global.parameters()) + list(D_local.parameters()),
        lr=2e-4, betas=(0.5, 0.999)
    )
    
    history = {'G_loss': [], 'D_loss': []}
    
    for epoch in range(epochs):
        for i, (real_imgs, labels) in enumerate(dataloader):
            real_imgs, labels = real_imgs.to(device), labels.to(device)
            batch = real_imgs.size(0)
            
            # ---- 1. 训练判别器 ----
            opt_D.zero_grad()
            
            # 生成假样本
            z = torch.randn(batch, latent_dim, device=device)
            fake_imgs = G(z, labels).detach()  # detach: 不传播到 G
            
            # 获取局部 patch（随机裁剪 16×16）
            _, _, h, w = real_imgs.shape
            top = torch.randint(0, h - 16 + 1, (1,)).item() if h > 16 else 0
            left = torch.randint(0, w - 16 + 1, (1,)).item() if w > 16 else 0
            real_patches = real_imgs[:, :, top:top+16, left:left+16]
            fake_patches = fake_imgs[:, :, top:top+16, left:left+16]
            
            # 判别器输出
            Dg_real = D_global(real_imgs, labels)
            Dg_fake = D_global(fake_imgs, labels)
            Dl_real = D_local(real_patches, labels)
            Dl_fake = D_local(fake_patches, labels)
            
            loss_D = criterion.discriminator_loss(Dg_real, Dg_fake, Dl_real, Dl_fake)
            loss_D.backward()
            opt_D.step()
            
            # ---- 2. 训练生成器 ----
            opt_G.zero_grad()
            
            z = torch.randn(batch, latent_dim, device=device)
            fake_imgs = G(z, labels)
            
            # 局部 patch
            _, _, h, w = fake_imgs.shape
            top2 = torch.randint(0, max(h-16+1, 1), (1,)).item()
            left2 = torch.randint(0, max(w-16+1, 1), (1,)).item()
            fake_patches = fake_imgs[:, :, top2:top2+16, left2:left2+16]
            
            Dg_fake = D_global(fake_imgs, labels)
            Dl_fake = D_local(fake_patches, labels)
            
            loss_G = criterion.generator_loss(Dg_fake, Dl_fake)
            loss_G.backward()
            opt_G.step()
            
            if i % 150 == 0:
                history['D_loss'].append(loss_D.item())
                history['G_loss'].append(loss_G.item())
        
        print(f"Epoch {epoch+1:2d}/{epochs} | D Loss: {loss_D.item():.4f} | G Loss: {loss_G.item():.4f}")
    
    return G, D_global, D_local, history

# 运行训练（CPU 模式下较快完成）
device_train = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
G_trained, _, _, history = train_ddcgan_demo(epochs=5, batch_size=64, device=device_train)


In [ ]:
# === DDcGAN 生成结果可视化 ===
@torch.no_grad()
def generate_and_show(G, num_classes=10, samples_per_class=8, device='cpu'):
    G.eval()
    latent_dim = G.latent_dim
    
    fig, axes = plt.subplots(num_classes, samples_per_class, figsize=(samples_per_class, num_classes))
    
    for cls in range(num_classes):
        z = torch.randn(samples_per_class, latent_dim, device=device)
        labels = torch.full((samples_per_class,), cls, dtype=torch.long, device=device)
        fake_imgs = G(z, labels).cpu()
        
        for j in range(samples_per_class):
            img = fake_imgs[j].squeeze().numpy()
            img = (img + 1) / 2  # [-1,1] → [0,1]
            axes[cls, j].imshow(img, cmap='gray')
            axes[cls, j].axis('off')
            if j == 0:
                axes[cls, j].set_ylabel(f'Class {cls}', fontsize=10, rotation=0, labelpad=20)
    
    plt.suptitle('DDcGAN Generated MNIST Digits (5 epochs)', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.savefig('/home/yisan/ai-learning/notebooks/ddcgan_generated.png', dpi=100, bbox_inches='tight')
    plt.show()

generate_and_show(G_trained, device=device_train)

# 训练曲线
if history['G_loss']:
    fig, ax = plt.subplots(figsize=(8, 4))
    steps = range(len(history['G_loss']))
    ax.plot(steps, history['D_loss'], label='D Loss', alpha=0.8)
    ax.plot(steps, history['G_loss'], label='G Loss', alpha=0.8)
    ax.set_xlabel('Training Steps (×150)'); ax.set_ylabel('Loss')
    ax.set_title('DDcGAN Training Curves (Hinge Loss)')
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('/home/yisan/ai-learning/notebooks/ddcgan_training.png', dpi=100, bbox_inches='tight')
    plt.show()


## 7.9 DDcGAN 关键设计要点

### 1. 投影判别器 (Projection Discriminator)

DDcGAN 使用**投影判别器**来注入条件信息，而非简单拼接：

$$
D(\mathbf{x}, c) = \mathbf{w}_c^T \phi(\mathbf{x}) + \psi(\phi(\mathbf{x}))
$$

其中 $\phi(\mathbf{x})$ 是判别器对图像提取的特征，$\mathbf{w}_c$ 是类别 $c$ 的嵌入向量。
这种设计已被证明优于简单的条件拼接。

### 2. 局部判别器的随机裁剪

每次前向传播时从图像中**随机裁剪**一个固定大小的 patch（如 16×16），
迫使局部判别器学会评估任意位置的纹理质量。

### 3. 架构不对称设计

| 组件 | 全局判别器 | 局部判别器 |
|------|-----------|-----------|
| 网络深度 | 4 层卷积 | 3 层卷积 |
| 特征维度 | 64 起始 → 512 最终 | 32 起始 → 128 最终 |
| 感受野 | 覆盖全图 (32×32) | 仅覆盖 patch (16×16) |
| 功能 | 判断结构一致性 | 判断纹理真实性 |

这种不对称设计是关键——两个判别器学到的特征**互补而非冗余**。


## 本章总结

### PCA 要点

| 概念 | 关键公式/操作 |
|------|--------------|
| 协方差矩阵 | $\mathbf{C} = \frac{1}{n-1} \tilde{\mathbf{X}}^T \tilde{\mathbf{X}}$ |
| 特征值分解 | $\mathbf{C} = \mathbf{V} \mathbf{\Lambda} \mathbf{V}^T$ |
| 降维 | $\mathbf{Z} = \tilde{\mathbf{X}} \mathbf{W}_k$ |
| 方差解释率 | $\lambda_i / \sum \lambda_j$ |

### DDcGAN 要点

| 组件 | 功能 |
|------|------|
| Generator | 噪声 + 条件 → 逼真图像 |
| Global Discriminator | 评估整体结构和全局一致性 |
| Local Discriminator | 评估局部纹理和细节质量 |
| Hinge Loss | 比交叉熵更稳定的对抗损失 |
| Projection Discriminator | 高效的条件信息注入 |
| 双判别器加权 | $\mathcal{L}_{D}^{\text{total}} = \mathcal{L}_{D_g} + \lambda \mathcal{L}_{D_l}$ |

✅ 第十章完成！你已深入理解 PCA 的数学原理和 DDcGAN 的架构设计。
